In [1]:
import os
import glob
import torch
from google.colab import drive

# 1. Hardware check
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemError("GPU not found! Go to Runtime -> Change runtime type -> Select T4 GPU.")

# 2. Mount Drive (if not already mounted via UI)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# Set confirmed path
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ELEC5622_Project2"

# 3. Setup paths on Colab scratch SSD (matching dataset.py names!)
DATA_DIR = "/content/dataset"
os.makedirs(f"{DATA_DIR}/training", exist_ok=True)
os.makedirs(f"{DATA_DIR}/validation", exist_ok=True)
os.makedirs(f"{DATA_DIR}/test", exist_ok=True)

# 4. Copy zip and CSV to local scratch SSD
print("Copying zip and CSV to local scratch SSD...")
!cp "{DRIVE_PROJECT_DIR}/Full_Data_Project2.zip" /content/
!cp "{DRIVE_PROJECT_DIR}/gt_training.csv" /content/dataset/

# 5. Unpack outer archive
!unzip -q -o /content/Full_Data_Project2.zip -d /content/temp_extract/

# 6. Unzip each split directly into /content/dataset
# (The inner zips already contain 'training/', 'validation/', and 'test/' root folders)
def find_and_unzip(keyword):
    matches = glob.glob(f"/content/temp_extract/**/*{keyword}*.zip", recursive=True)
    if matches:
        zip_path = matches[0]
        print(f"Extracting {os.path.basename(zip_path)} -> /content/dataset...")
        !unzip -q -o "{zip_path}" -d /content/dataset/
    else:
        print(f"Warning: No zip file matching '*{keyword}*.zip' found!")

find_and_unzip("train")
find_and_unzip("val")
find_and_unzip("test")

# If the CSV was inside the zip, copy to /content/dataset/
!find /content/temp_extract -name "*.csv" -exec cp {} /content/dataset/ \;

# 7. Clean up temporary archives to preserve disk space
!rm -rf /content/temp_extract /content/Full_Data_Project2.zip

# 8. Sanity check image counts
train_count = len(glob.glob(f"{DATA_DIR}/training/**/*.png", recursive=True))
val_count   = len(glob.glob(f"{DATA_DIR}/validation/**/*.png", recursive=True))
test_count  = len(glob.glob(f"{DATA_DIR}/test/**/*.png", recursive=True))

print("\nExtraction Complete!")
print(f"Train images: {train_count} (Expected: 8,701)")
print(f"Val images:   {val_count}   (Expected: 2,175)")
print(f"Test images:  {test_count}  (Expected: 2,720)")

CUDA Available: True
GPU: Tesla T4
Mounted at /content/drive
Copying zip and CSV to local scratch SSD...
Extracting training.zip -> /content/dataset...
Extracting validation.zip -> /content/dataset...
Extracting test.zip -> /content/dataset...

Extraction Complete!
Train images: 8701 (Expected: 8,701)
Val images:   2175   (Expected: 2,175)
Test images:  2720  (Expected: 2,720)


In [2]:
import os

repo_url = "https://github.com/juliuslehner/ELEC5622_Project2.git"

if not os.path.exists('/content/repo'):
    !git clone {repo_url} /content/repo
else:
    !git -C /content/repo pull

%cd /content/repo

Cloning into '/content/repo'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 34 (delta 11), reused 28 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 13.91 KiB | 6.95 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/repo


In [ ]:
!python main.py \
    --arch alexnet \
    --strategy partial \
    --epochs 25 \
    --lr 1e-4 \
    --batch_size 64 \
    --data_dir /content/dataset \
    --csv_path /content/dataset/gt_training.csv \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs

Using device: cuda
Loading data from '/content/dataset'...
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100% 233M/233M [00:01<00:00, 188MB/s]
[ALEXNET | PARTIAL] Trainable params: 54,558,726 / 57,028,422 (95.67%)

Training on cuda for 25 epochs...
------------------------------------------------------------------------
Epoch 01/25 | Train Loss: 0.5737 Acc: 78.50% | Val Loss: 0.3169 Acc: 87.86% | Time: 27.6s [* Best Model Saved]
Epoch 02/25 | Train Loss: 0.3444 Acc: 87.27% | Val Loss: 0.2738 Acc: 90.30% | Time: 24.7s [* Best Model Saved]
Epoch 03/25 | Train Loss: 0.3084 Acc: 88.73% | Val Loss: 0.2218 Acc: 92.78% | Time: 28.8s [* Best Model Saved]
Epoch 04/25 | Train Loss: 0.2695 Acc: 90.00% | Val Loss: 0.2230 Acc: 91.59% | Time: 28.9s
Epoch 05/25 | 

In [ ]:
!python main.py \
    --arch alexnet \
    --strategy full \
    --epochs 25 \
    --lr 1e-4 \
    --batch_size 64 \
    --data_dir /content/dataset \
    --csv_path /content/dataset/gt_training.csv \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs

Using device: cuda
Loading data from '/content/dataset'...
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
[ALEXNET | FULL] Trainable params: 57,028,422 / 57,028,422 (100.00%)

Training on cuda for 25 epochs...
------------------------------------------------------------------------
Epoch 01/25 | Train Loss: 0.4481 Acc: 83.25% | Val Loss: 0.2288 Acc: 92.05% | Time: 25.9s [* Best Model Saved]
Epoch 02/25 | Train Loss: 0.2456 Acc: 91.07% | Val Loss: 0.1922 Acc: 92.92% | Time: 25.3s [* Best Model Saved]
Epoch 03/25 | Train Loss: 0.1979 Acc: 93.15% | Val Loss: 0.1573 Acc: 94.76% | Time: 31.0s [* Best Model Saved]
Epoch 04/25 | Train Loss: 0.1787 Acc: 93.62% | Val Loss: 0.1731 Acc: 93.98% | Time: 30.8s
Epoch 05/25 | Train Loss: 0.1599 Acc: 94.35% | Val Loss: 0.1624 Acc: 94.48% | Time: 26.2s
Epoch 06/25 | Train Loss: 0.1535 Acc: 94.60% | Val Loss: 0.1525 Acc: 94.99% | Time: 24.8s [* Best Model

In [4]:
!python main.py \
    --arch resnet34 \
    --strategy partial \
    --epochs 25 \
    --lr 1e-4 \
    --batch_size 64 \
    --data_dir /content/dataset \
    --csv_path /content/dataset/gt_training.csv \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs

Using device: cuda
Loading data from '/content/dataset'...
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 150MB/s]
[RESNET34 | PARTIAL] Trainable params: 13,117,446 / 21,287,750 (61.62%)

Training on cuda for 25 epochs...
------------------------------------------------------------------------
Epoch 01/25 | Train Loss: 0.4061 Acc: 85.92% | Val Loss: 0.2420 Acc: 91.59% | Time: 32.2s [* Best Model Saved]
Epoch 02/25 | Train Loss: 0.2376 Acc: 91.60% | Val Loss: 0.1948 Acc: 93.89% | Time: 28.8s [* Best Model Saved]
Epoch 03/25 | Train Loss: 0.1912 Acc: 93.25% | Val Loss: 0.2131 Acc: 92.60% | Time: 30.6s
Epoch 04/25 | Train Loss: 0.1740 Acc: 94.05% | Val Loss: 0.1421 Acc: 95.36% | Time: 28.6s [* Best Model Saved]
Epoch 05/25 | Tra

In [5]:
!python main.py \
    --arch resnet34 \
    --strategy full \
    --epochs 25 \
    --lr 1e-4 \
    --batch_size 64 \
    --data_dir /content/dataset \
    --csv_path /content/dataset/gt_training.csv \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs

Using device: cuda
Loading data from '/content/dataset'...
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
[RESNET34 | FULL] Trainable params: 21,287,750 / 21,287,750 (100.00%)

Training on cuda for 25 epochs...
------------------------------------------------------------------------
Epoch 01/25 | Train Loss: 0.3655 Acc: 87.05% | Val Loss: 0.2541 Acc: 91.13% | Time: 52.1s [* Best Model Saved]
Epoch 02/25 | Train Loss: 0.1879 Acc: 93.70% | Val Loss: 0.1708 Acc: 94.16% | Time: 50.8s [* Best Model Saved]
Epoch 03/25 | Train Loss: 0.1384 Acc: 95.38% | Val Loss: 0.0911 Acc: 97.20% | Time: 51.3s [* Best Model Saved]
Epoch 04/25 | Train Loss: 0.1124 Acc: 96.07% | Val Loss: 0.1414 Acc: 94.44% | Time: 51.2s
Epoch 05/25 | Train Loss: 0.1074 Acc: 96.43% | Val Loss: 0.1290 Acc: 96.00% | Time: 51.2s
Epoch 06/25 | Train Loss: 0.0984 Acc: 96.67% | Val Loss: 0.1512 Acc: 95.54% | Time: 51.3s
Epoch 07/25 

In [ ]:
!python repo/evaluate.py \
    --arch alexnet \
    --strategy full \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs/alexnet_full

EVALUATION SUITE: ALEXNET [FULL]
Using device: cuda
Output directory: /content/drive/MyDrive/ELEC5622_Project2/outputs/alexnet_full
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
Loading weights from: /content/drive/MyDrive/ELEC5622_Project2/outputs/alexnet_full/best_model.pth

PER-CLASS CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Centromere     0.9678    0.9877    0.9776       487
       Golgi     0.9565    0.9333    0.9448       165
 Homogeneous     0.9177    0.9809    0.9482       523
       NuMem     0.9863    0.9819    0.9841       441
   Nucleolar     0.9658    0.9622    0.9640       529
    Speckled     0.9702    0.9061    0.9371       575

    accuracy                         0.9599      2720
   macro avg     0.9607    0.9587    0.9593      2720
weighted avg     0.9606    0.9599    0.9598      2720

Confusion matrix plot successfully saved to:

In [ ]:
!python repo/evaluate.py \
    --arch alexnet \
    --strategy partial \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs/alexnet_partial

EVALUATION SUITE: ALEXNET [PARTIAL]
Using device: cuda
Output directory: /content/drive/MyDrive/ELEC5622_Project2/outputs/alexnet_partial
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
Loading weights from: /content/drive/MyDrive/ELEC5622_Project2/outputs/alexnet_partial/best_model.pth

PER-CLASS CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Centromere     0.9652    0.9692    0.9672       487
       Golgi     0.9162    0.9273    0.9217       165
 Homogeneous     0.9564    0.9235    0.9397       523
       NuMem     0.9659    0.9637    0.9648       441
   Nucleolar     0.9450    0.9414    0.9432       529
    Speckled     0.9105    0.9374    0.9237       575

    accuracy                         0.9449      2720
   macro avg     0.9432    0.9437    0.9434      2720
weighted avg     0.9452    0.9449    0.9449      2720

Confusion matrix plot successfully 

In [6]:
!python evaluate.py \
    --arch resnet34 \
    --strategy partial \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs/resnet34_partial

EVALUATION SUITE: RESNET34 [PARTIAL]
Using device: cuda
Output directory: /content/drive/MyDrive/ELEC5622_Project2/outputs/resnet34_partial
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
Loading weights from: /content/drive/MyDrive/ELEC5622_Project2/outputs/resnet34_partial/best_model.pth

PER-CLASS CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Centromere     0.9796    0.9836    0.9816       487
       Golgi     0.9506    0.9333    0.9419       165
 Homogeneous     0.9583    0.9656    0.9619       523
       NuMem     0.9730    0.9819    0.9774       441
   Nucleolar     0.9550    0.9622    0.9586       529
    Speckled     0.9574    0.9391    0.9482       575

    accuracy                         0.9632      2720
   macro avg     0.9623    0.9609    0.9616      2720
weighted avg     0.9632    0.9632    0.9632      2720

Confusion matrix plot successful

In [7]:
!python evaluate.py \
    --arch resnet34 \
    --strategy full \
    --output_dir /content/drive/MyDrive/ELEC5622_Project2/outputs/resnet34_full

EVALUATION SUITE: RESNET34 [FULL]
Using device: cuda
Output directory: /content/drive/MyDrive/ELEC5622_Project2/outputs/resnet34_full
Loaded 8701 images from /content/dataset/training
Loaded 2175 images from /content/dataset/validation
Loaded 2720 images from /content/dataset/test
Loading weights from: /content/drive/MyDrive/ELEC5622_Project2/outputs/resnet34_full/best_model.pth

PER-CLASS CLASSIFICATION REPORT:
              precision    recall  f1-score   support

  Centromere     0.9856    0.9856    0.9856       487
       Golgi     0.9876    0.9636    0.9755       165
 Homogeneous     0.9864    0.9713    0.9788       523
       NuMem     0.9712    0.9932    0.9821       441
   Nucleolar     0.9685    0.9868    0.9775       529
    Speckled     0.9665    0.9530    0.9597       575

    accuracy                         0.9761      2720
   macro avg     0.9776    0.9756    0.9765      2720
weighted avg     0.9762    0.9761    0.9761      2720

Confusion matrix plot successfully saved 